In [13]:
!pip install google-search-results


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from serpapi import GoogleSearch
import json

API_KEY = "3c67553c1f053f62236691701d6ded2ffe3162489f69971f8272f441ec7b20f1"

params = {
    "engine": "google_maps",
    "q": "KFC Madurai",
    "api_key": API_KEY
}

search = GoogleSearch(params)
results = search.get_dict()

print(json.dumps(results, indent=2))

{
  "search_metadata": {
    "id": "6a0e20cef5d5ac556e85ef7d",
    "status": "Success",
    "json_endpoint": "https://serpapi.com/searches/ZaJrdy6Omy5D2RUJpXUEYQ/6a0e20cef5d5ac556e85ef7d.json",
    "created_at": "2026-05-20 20:59:58 UTC",
    "processed_at": "2026-05-20 20:59:58 UTC",
    "google_maps_url": "https://www.google.com/maps/search/KFC+Madurai//?hl=en",
    "raw_html_file": "https://serpapi.com/searches/ZaJrdy6Omy5D2RUJpXUEYQ/6a0e20cef5d5ac556e85ef7d.html",
    "prettify_html_file": "https://serpapi.com/searches/ZaJrdy6Omy5D2RUJpXUEYQ/6a0e20cef5d5ac556e85ef7d.prettify",
    "total_time_taken": 1.07
  },
  "search_parameters": {
    "engine": "google_maps",
    "type": "search",
    "q": "KFC Madurai",
    "google_domain": "google.com",
    "hl": "en"
  },
  "search_information": {
    "local_results_state": "Results for exact spelling",
    "query_displayed": "KFC Madurai"
  },
  "local_results": [
    {
      "position": 1,
      "title": "KFC",
      "place_id": "ChIJS6mAn

In [15]:
from serpapi import GoogleSearch
import pandas as pd
import json
import time

API_KEY = "3c67553c1f053f62236691701d6ded2ffe3162489f69971f8272f441ec7b20f1"

# 5 businesses in Madurai
businesses = [
    "KFC Madurai",
    "Domino's Pizza Madurai",
    "A2B Madurai",
    "McDonald's Madurai",
    "Pizza Hut Madurai"
]

all_reviews = []

for business in businesses:
    print(f"\nFetching reviews for {business}...")

    try:
        # Step 1: Search business and get data_id
        search = GoogleSearch({
            "engine": "google_maps",
            "q": business,
            "api_key": API_KEY
        })

        results = search.get_dict()

        if "local_results" in results and len(results["local_results"]) > 0:
            data_id = results["local_results"][0]["data_id"]
        elif "place_results" in results:
            data_id = results["place_results"]["data_id"]
        else:
            print(f"No data_id found for {business}")
            continue

        print("Found data_id:", data_id)

        # Step 2: Collect max 100 reviews for this business
        business_reviews = []
        next_page_token = None

        while len(business_reviews) < 100:
            params = {
                "engine": "google_maps_reviews",
                "data_id": data_id,
                "api_key": API_KEY,
                "hl": "en"
            }

            if next_page_token:
                params["next_page_token"] = next_page_token

            review_search = GoogleSearch(params)
            review_results = review_search.get_dict()

            reviews = review_results.get("reviews", [])

            if not reviews:
                break

            for r in reviews:
                business_reviews.append({
                    "Business_Name": business,
                    "User": r.get("user", {}).get("name"),
                    "Rating": r.get("rating"),
                    "Date": r.get("date"),
                    "Review_Text": r.get("snippet")
                })

            print(f"Collected {len(business_reviews)} reviews for {business}")

            # Next page
            next_page_token = review_results.get(
                "serpapi_pagination", {}
            ).get("next_page_token")

            if not next_page_token:
                break

            time.sleep(2)  # avoid API rate limits

        # Keep max 100 per business
        all_reviews.extend(business_reviews[:100])

        time.sleep(2)

    except Exception as e:
        print(f"Error with {business}: {e}")

# Step 3: Convert to DataFrame
df = pd.DataFrame(all_reviews)

# Save CSV
df.to_csv("madurai_reviews_500.csv", index=False)

# Save JSON
with open("madurai_reviews_500.json", "w", encoding="utf-8") as f:
    json.dump(all_reviews, f, indent=4, ensure_ascii=False)

print("\n==============================")
print("Total Reviews Collected:", len(df))
print("Saved: madurai_reviews_500.csv")
print("Saved: madurai_reviews_500.json")
print("==============================")


Fetching reviews for KFC Madurai...
Found data_id: 0x3b00cff39c80a94b:0xb2d3b54e43cd0f1c
Collected 8 reviews for KFC Madurai
Collected 18 reviews for KFC Madurai
Collected 28 reviews for KFC Madurai
Collected 38 reviews for KFC Madurai
Collected 48 reviews for KFC Madurai
Collected 58 reviews for KFC Madurai
Collected 68 reviews for KFC Madurai
Collected 78 reviews for KFC Madurai
Collected 88 reviews for KFC Madurai
Collected 98 reviews for KFC Madurai
Collected 108 reviews for KFC Madurai

Fetching reviews for Domino's Pizza Madurai...
Found data_id: 0x3b00cf73d379ba77:0xdc70f96014b12d43
Collected 8 reviews for Domino's Pizza Madurai
Collected 18 reviews for Domino's Pizza Madurai
Collected 28 reviews for Domino's Pizza Madurai
Collected 38 reviews for Domino's Pizza Madurai
Collected 48 reviews for Domino's Pizza Madurai
Collected 58 reviews for Domino's Pizza Madurai
Collected 68 reviews for Domino's Pizza Madurai
Collected 78 reviews for Domino's Pizza Madurai
Collected 88 review

In [17]:
import pandas as pd
import re

# Load dataset
df = pd.read_json("madurai_reviews_500.json")

# -----------------------------
# 1. Remove duplicates
# -----------------------------
df.drop_duplicates(subset=["Review_Text"], inplace=True)

# -----------------------------
# 2. Remove missing values
# -----------------------------
df.dropna(subset=["Review_Text"], inplace=True)

# -----------------------------
# 3. Convert to lowercase
# -----------------------------
df["Clean_Text"] = df["Review_Text"].str.lower()

# -----------------------------
# 4. Remove special chars / numbers
# -----------------------------
df["Clean_Text"] = df["Clean_Text"].apply(
    lambda x: re.sub(r'[^a-zA-Z\s]', '', str(x))
)

# -----------------------------
# 5. Simple tokenization
# -----------------------------
df["Tokens"] = df["Clean_Text"].apply(lambda x: x.split())

# -----------------------------
# 6. Stopword removal (manual list)
# -----------------------------
stop_words = {
    "the","is","a","an","and","or","to","of","in","on",
    "for","with","at","this","that","it","was","are"
}

df["Filtered_Tokens"] = df["Tokens"].apply(
    lambda words: [w for w in words if w not in stop_words]
)

# -----------------------------
# 7. Join cleaned text
# -----------------------------
df["Processed_Text"] = df["Filtered_Tokens"].apply(
    lambda words: " ".join(words)
)

# -----------------------------
# 8. Sentiment label from rating
# -----------------------------
def sentiment_label(rating):
    if rating >= 4:
        return "Positive"
    elif rating == 3:
        return "Neutral"
    else:
        return "Negative"

df["Sentiment_Label"] = df["Rating"].apply(sentiment_label)

# -----------------------------
# 9. Review length
# -----------------------------
df["Review_Length"] = df["Processed_Text"].apply(
    lambda x: len(str(x).split())
)

# Final structured dataset
final_df = df[
    [
        "Business_Name",
        "User",
        "Rating",
        "Date",
        "Review_Text",
        "Processed_Text",
        "Sentiment_Label",
        "Review_Length"
    ]
]

# Save files
final_df.to_csv("cleaned_reviews.csv", index=False)
final_df.to_json(
    "cleaned_reviews.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("Total Clean Reviews:", len(final_df))
print(final_df.head())

Total Clean Reviews: 498
  Business_Name                        User  Rating          Date  \
0   KFC Madurai             Mohammed sameer       3  3 months ago   
1   KFC Madurai                      Aswath       3  6 months ago   
2   KFC Madurai              Pradeep Sanjay       1  4 months ago   
3   KFC Madurai               Vetri Venthan       5   2 years ago   
4   KFC Madurai  Saravana Kumar Subramanian       5  3 months ago   

                                         Review_Text  \
0  The food here is okay, but only really good on...   
1  Ordering experience was easy and the KFC chick...   
2  Very very worst food the chicken served in the...   
3  I had an absolutely delightful dining experien...   
4  We prefer the previous Classic Zinger burger w...   

                                      Processed_Text Sentiment_Label  \
0  food here okay but only really good wednesdays...         Neutral   
1  ordering experience easy kfc chicken decent en...         Neutral   
2  very

In [18]:
!pip install transformers torch pandas

   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.8 MB 2.8 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/10.8 MB 2.7 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/10.8 MB 3.0 MB/s eta 0:00:03
   ---------- ----------------------------- 2.9/10.8 MB 3.4 MB/s eta 0:00:03
   ------------- -------------------------- 3.7/10.8 MB 3.3 MB/s eta 0:00:03
   ---------------- ----------------------- 4.5/10.8 MB 3.6 MB/s eta 0:00:02
   ------------------- -------------------- 5.2/10.8 MB 3.6 MB/s eta 0:00:02
   ---------------------- ----------------- 6.0/10.8 MB 3.5 MB/s eta 0:00:02
   -------------------------- ------------- 7.1/10.8 MB 3.7 MB/s eta 0:00:02
   ------------------------------ --------- 8.1/10.8 MB 3.9 MB/s eta 0:00:01
   ---------------------------------- ----- 9.2/10.8 MB 4.0 MB/s eta 0:00:01
   ----------


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline
import pandas as pd

# Load dataset
df = pd.read_json("cleaned_reviews.json")

# Load sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True
)

# Keep reviews shorter (extra safety)
df["Short_Text"] = df["Processed_Text"].astype(str).str[:400]

# Run sentiment
df["LLM_Sentiment"] = df["Short_Text"].apply(
    lambda x: sentiment_model(x)[0]["label"]
)

print(df[["Short_Text", "LLM_Sentiment"]].head())

# Save
df.to_json("reviews_with_sentiment.json", orient="records", indent=4)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20202.13it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                                          Short_Text LLM_Sentiment
0  food here okay but only really good wednesdays...      positive
1  ordering experience easy kfc chicken decent en...      negative
2  very very worst food chicken served wednesday ...      negative
3  i had absolutely delightful dining experience ...      positive
4  we prefer previous classic zinger burger white...      positive


In [3]:
import pandas as pd

# Load dataset with sentiment
df = pd.read_json("reviews_with_sentiment.json")

# Topic extraction function
def extract_topic(text):
    text = str(text).lower()

    if any(w in text for w in ["staff", "employee", "manager", "service"]):
        return "Service/Staff"

    elif any(w in text for w in ["price", "cost", "expensive", "cheap"]):
        return "Pricing"

    elif any(w in text for w in ["clean", "dirty", "hygiene"]):
        return "Cleanliness"

    elif any(w in text for w in ["delivery", "late", "time", "wait", "queue"]):
        return "Delivery/Waiting"

    elif any(w in text for w in ["food", "taste", "burger", "chicken", "meal"]):
        return "Food Quality"

    elif any(w in text for w in ["music", "crowd", "seat", "ambience", "atmosphere"]):
        return "Ambience"

    else:
        return "General"

# Apply topic extraction
df["Topic"] = df["Processed_Text"].apply(extract_topic)

print(df[["Processed_Text", "LLM_Sentiment", "Topic"]].head())

# Save updated file
df.to_json("reviews_with_topics.json", orient="records", indent=4)

                                      Processed_Text LLM_Sentiment  \
0  food here okay but only really good wednesdays...      positive   
1  ordering experience easy kfc chicken decent en...      negative   
2  very very worst food chicken served wednesday ...      negative   
3  i had absolutely delightful dining experience ...      positive   
4  we prefer previous classic zinger burger white...      positive   

           Topic  
0  Service/Staff  
1   Food Quality  
2   Food Quality  
3  Service/Staff  
4   Food Quality  


In [4]:
import pandas as pd

df = pd.read_json("reviews_with_topics.json")

# Sentiment count
print("\n===== SENTIMENT SUMMARY =====")
print(df["LLM_Sentiment"].value_counts())

# Topic count
print("\n===== TOPIC SUMMARY =====")
print(df["Topic"].value_counts())

# Negative reviews analysis
negative_reviews = df[df["LLM_Sentiment"].str.lower() == "negative"]

print("\n===== TOP NEGATIVE TOPICS =====")
print(negative_reviews["Topic"].value_counts())

# Positive reviews analysis
positive_reviews = df[df["LLM_Sentiment"].str.lower() == "positive"]

print("\n===== TOP POSITIVE TOPICS =====")
print(positive_reviews["Topic"].value_counts())


===== SENTIMENT SUMMARY =====
LLM_Sentiment
negative    246
positive    225
neutral      27
Name: count, dtype: int64

===== TOPIC SUMMARY =====
Topic
Service/Staff       214
Food Quality        103
Delivery/Waiting     69
General              58
Pricing              27
Cleanliness          23
Ambience              4
Name: count, dtype: int64

===== TOP NEGATIVE TOPICS =====
Topic
Service/Staff       97
Food Quality        49
Delivery/Waiting    45
General             25
Cleanliness         14
Pricing             13
Ambience             3
Name: count, dtype: int64

===== TOP POSITIVE TOPICS =====
Topic
Service/Staff       110
Food Quality         46
General              30
Delivery/Waiting     19
Pricing              11
Cleanliness           8
Ambience              1
Name: count, dtype: int64


In [5]:
df.to_csv("final_reviews_analysis.csv", index=False)
df.to_json("final_reviews_analysis.json", orient="records", indent=4)
print("Saved successfully")

Saved successfully


In [6]:
import pandas as pd

df = pd.read_json("final_reviews_analysis.json")

# Negative reviews
negative_reviews = df[
    df["LLM_Sentiment"].astype(str).str.lower() == "negative"
]

negative_topics = negative_reviews["Topic"].value_counts()

print("===== OPERATIONAL IMPROVEMENT SUGGESTIONS =====\n")

for topic, count in negative_topics.items():
    print(f"{topic}: {count} complaints")

    if topic == "Service/Staff":
        print("- Recommendation: Train staff response time and improve customer interaction.\n")

    elif topic == "Pricing":
        print("- Recommendation: Re-evaluate pricing, combo offers, or perceived value.\n")

    elif topic == "Delivery/Waiting":
        print("- Recommendation: Reduce queue/wait times and improve delivery workflow.\n")

    elif topic == "Cleanliness":
        print("- Recommendation: Improve hygiene checks and store cleanliness.\n")

    elif topic == "Food Quality":
        print("- Recommendation: Improve consistency in taste, freshness, and serving quality.\n")

    elif topic == "Ambience":
        print("- Recommendation: Improve seating, comfort, and in-store environment.\n")

    else:
        print("- Recommendation: Investigate recurring general customer issues.\n")

===== OPERATIONAL IMPROVEMENT SUGGESTIONS =====

Service/Staff: 97 complaints
- Recommendation: Train staff response time and improve customer interaction.

Food Quality: 49 complaints
- Recommendation: Improve consistency in taste, freshness, and serving quality.

Delivery/Waiting: 45 complaints
- Recommendation: Reduce queue/wait times and improve delivery workflow.

General: 25 complaints
- Recommendation: Investigate recurring general customer issues.

Cleanliness: 14 complaints
- Recommendation: Improve hygiene checks and store cleanliness.

Pricing: 13 complaints
- Recommendation: Re-evaluate pricing, combo offers, or perceived value.

Ambience: 3 complaints
- Recommendation: Improve seating, comfort, and in-store environment.



In [7]:
import pandas as pd

df = pd.read_json("final_reviews_analysis.json")

print("\n===== TREND-BASED BUSINESS RECOMMENDATIONS =====\n")

sentiment_counts = df["LLM_Sentiment"].astype(str).str.lower().value_counts()

print(sentiment_counts)

positive = sentiment_counts.get("positive", 0)
negative = sentiment_counts.get("negative", 0)
neutral = sentiment_counts.get("neutral", 0)

if negative > positive:
    print("\nBusiness Trend: Customer dissatisfaction is high.")
    print("Recommendation: Prioritize complaint-heavy areas immediately.")

elif positive > negative:
    print("\nBusiness Trend: Overall customer sentiment is positive.")
    print("Recommendation: Maintain strengths and optimize weaker categories.")

else:
    print("\nBusiness Trend: Mixed sentiment.")
    print("Recommendation: Improve consistency across customer experience.")


===== TREND-BASED BUSINESS RECOMMENDATIONS =====

LLM_Sentiment
negative    246
positive    225
neutral      27
Name: count, dtype: int64

Business Trend: Customer dissatisfaction is high.
Recommendation: Prioritize complaint-heavy areas immediately.


In [8]:
import pandas as pd

df = pd.read_json("final_reviews_analysis.json")

print("\n===== COMPETITOR COMPARISON =====\n")

comparison = df.groupby("Business_Name").agg(
    Avg_Rating=("Rating", "mean"),
    Total_Reviews=("Review_Text", "count")
)

print(comparison.sort_values("Avg_Rating", ascending=False))


===== COMPETITOR COMPARISON =====

                        Avg_Rating  Total_Reviews
Business_Name                                    
Pizza Hut Madurai         4.163265             98
McDonald's Madurai        3.330000            100
A2B Madurai               3.060000            100
KFC Madurai               2.510000            100
Domino's Pizza Madurai    2.110000            100


In [9]:
df.to_csv("business_recommendation_output.csv", index=False)
df.to_json(
    "business_recommendation_output.json",
    orient="records",
    indent=4
)

print("Saved recommendation-ready dataset")

Saved recommendation-ready dataset
